# 🚀 Boosting Models — PII Data Detection

Notebook ini menjalankan training & evaluasi **XGBoost** dan **LightGBM** untuk token-level PII detection.

**4 varian model:**
1. LightGBM Imbalance (data penuh ~3M tokens)
2. XGBoost Imbalance (data penuh ~3M tokens)
3. LightGBM Balance (undersample O + oversample PII)
4. XGBoost Balance (undersample O + oversample PII)

**Output:**
- 4 CSV predictions di `results/predictions/`
- 4 JSON metrics di `results/metrics/`

> ⚠️ **Pastikan GPU accelerator ENABLED di Kaggle settings (untuk XGBoost CUDA)**

## Cell 1 — Clone Repository & Install Dependencies

In [ ]:
import shutil, os
shutil.rmtree("/kaggle/working/PII-Data-Detection", ignore_errors=True)
!git clone -b feature/boosting-models https://github.com/yukakeren/PII-Data-Detection.git
%cd PII-Data-Detection

In [ ]:
!grep -v "pycrfsuite" requirements.txt > requirements_fixed.txt
!pip install -r requirements_fixed.txt -q
!pip install -r models/boosting/requirements.txt -q

## Cell 2 — Copy Dataset ke data/processed/

> ⚠️ Sesuaikan path dataset Kaggle kamu di bawah ini

In [ ]:
import os

# Buat direktori yang diperlukan
for d in ["data/processed", "results/predictions", "results/metrics"]:
    os.makedirs(d, exist_ok=True)

# Copy dataset dari Kaggle input
# ⚠️ SESUAIKAN PATH INI dengan dataset Kaggle kamu
!cp /kaggle/input/datasets/ericatriana/pii-data-train/train.json data/processed/
!cp /kaggle/input/datasets/ericatriana/pii-test-data/test.json data/processed/ 2>/dev/null || echo "⚠️ test.json belum ditemukan"

!ls -la data/processed/

## Cell 3 — Import Modules & Load Data

In [ ]:
import os, sys, warnings
import numpy as np
warnings.filterwarnings("ignore")

# Setup path
sys.path.insert(0, "/kaggle/working/PII-Data-Detection")
os.chdir("/kaggle/working/PII-Data-Detection")

# Import project modules
from src.data_loader import DataLoader
from models.boosting.feature_extraction import (
    build_feature_matrix, encode_labels,
    compute_class_weights, get_sample_weights, quick_f1_pii,
)
from models.boosting.train_lightgbm import (
    train_lightgbm_imbalance, train_lightgbm_balance, balance_dataset,
)
from models.boosting.train_xgboost import (
    train_xgboost_imbalance, train_xgboost_balance, create_label_remap,
)
from models.boosting.predict import run_prediction_pipeline

print("✅ Semua module berhasil di-import!")
print(f"Working dir: {os.getcwd()}")

## Cell 4 — Load Data & Feature Extraction

> ⏱️ Proses ini memakan waktu ~5 menit (building features untuk ~3M tokens)

In [ ]:
# Load data
print("📂 Loading data...")
loader = DataLoader()
train_data = loader.load_raw_json("data/processed/train.json")
val_data   = loader.load_raw_json("data/processed/val.json")
test_data  = loader.load_raw_json("data/processed/test.json")
print(f"  Train: {len(train_data)} | Val: {len(val_data)} | Test: {len(test_data)}")

# Build features
print("\n🔧 Building features...")
X_train, y_train_str, _, vectorizer = build_feature_matrix(train_data, fit_vectorizer=True)
X_val,   y_val_str,   _, _          = build_feature_matrix(val_data,   vectorizer=vectorizer)
X_test,  y_test_str,  meta_test, _  = build_feature_matrix(test_data,  vectorizer=vectorizer)
print(f"  Feature shape: Train={X_train.shape}, Val={X_val.shape}, Test={X_test.shape}")

# Encode labels
all_labels = y_train_str + y_val_str + y_test_str
_, le = encode_labels(all_labels, fit=True)
y_train = le.transform(y_train_str)
y_val   = le.transform(y_val_str)
o_index = list(le.classes_).index("O")
print(f"  Label classes ({len(le.classes_)}): {list(le.classes_)}")
print(f"  O index: {o_index}")

## Cell 5 — Train LightGBM (Imbalance + Balance)

> ⏱️ Imbalance ~35 menit, Balance ~5 menit

In [ ]:
# ── LightGBM Imbalance ──
lgb_imb, lgb_imb_f1, sw_imb = train_lightgbm_imbalance(
    X_train, y_train, y_train_str, X_val, y_val, le
)

# ── Balance dataset ──
X_train_bal, y_train_bal, y_train_bal_str = balance_dataset(
    X_train, y_train, y_train_str
)

# ── LightGBM Balance ──
lgb_bal, lgb_bal_f1, _ = train_lightgbm_balance(
    X_train_bal, y_train_bal, y_train_bal_str, X_val, y_val, le
)

print(f"\n📊 LightGBM Summary:")
print(f"  Imbalance Val F1: {lgb_imb_f1:.4f}")
print(f"  Balance   Val F1: {lgb_bal_f1:.4f}")

## Cell 6 — Train XGBoost (Imbalance + Balance)

> ⏱️ Imbalance ~13 menit (GPU), Balance ~3 menit (GPU)

In [ ]:
# ── Label remapping (XGBoost butuh contiguous 0..N-1) ──
remap_imb, reverse_remap_imb, y_train_remapped = create_label_remap(y_train)

# ── XGBoost Imbalance ──
xgb_imb, xgb_imb_f1 = train_xgboost_imbalance(
    X_train, y_train, y_train_str, X_val, y_val, le,
    remap_imb, reverse_remap_imb, y_train_remapped, sw_imb,
)

# ── XGBoost Balance ──
xgb_bal, xgb_bal_f1, remap_bal, reverse_remap_bal = train_xgboost_balance(
    X_train_bal, y_train_bal, y_train_bal_str, X_val, y_val, le, o_index,
)

print(f"\n📊 XGBoost Summary:")
print(f"  Imbalance Val F1: {xgb_imb_f1:.4f}")
print(f"  Balance   Val F1: {xgb_bal_f1:.4f}")

## Cell 7 — Predict & Evaluate Semua Model

> ⏱️ ~20 menit (threshold tuning + prediction untuk 4 model)

In [ ]:
# Kumpulkan semua model
models_dict = {
    "lgb_imb": lgb_imb,
    "xgb_imb": xgb_imb,
    "lgb_bal": lgb_bal,
    "xgb_bal": xgb_bal,
}

# Jalankan pipeline prediksi lengkap
all_metrics = run_prediction_pipeline(
    models_dict=models_dict,
    X_test=X_test,
    X_val=X_val,
    y_val_str=y_val_str,
    meta_test=meta_test,
    le=le,
    o_index=o_index,
    remap_imb=remap_imb,
    reverse_remap_imb=reverse_remap_imb,
    remap_bal=remap_bal,
    reverse_remap_bal=reverse_remap_bal,
)

## Cell 8 — Verifikasi Output Files

In [ ]:
import os

print("📁 Predictions CSV:")
for f in sorted(os.listdir("results/predictions/")):
    if f.endswith(".csv"):
        size = os.path.getsize(f"results/predictions/{f}") / 1024 / 1024
        print(f"  ✅ {f} ({size:.1f} MB)")

print("\n📁 Metrics JSON:")
for f in sorted(os.listdir("results/metrics/")):
    if f.endswith(".json") and f != "metrics_template.json":
        print(f"  ✅ {f}")

# Verifikasi dengan script evaluasi project
print("\n🔍 Verifikasi dengan src/evaluate.py:")
from src.evaluate import evaluate_from_csv
for csv_file in sorted(os.listdir("results/predictions/")):
    if csv_file.endswith(".csv") and "boosting" not in csv_file:
        try:
            m = evaluate_from_csv(f"results/predictions/{csv_file}", csv_file.replace("_predictions.csv", ""))
            print(f"  ✅ {csv_file}: Token F1={m['token_level']['f1']:.4f}, Entity F1={m['entity_level']['f1']:.4f}")
        except Exception as e:
            print(f"  ❌ {csv_file}: {e}")

print("\n🎉 Semua output berhasil di-generate!")